# 🔢 PyTorch CNN — 손글씨 숫자(MNIST)를 합성곱 신경망으로

**건축공학딥러닝프로그래밍 · PyTorch CNN (MNIST) · 실습 노트북 (실습본 · 빈칸 채우기)**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/baekja/Lecture_DL-AE/blob/main/PyTorch_MNIST_CNN/mnist_cnn_practice.ipynb) ← **실습본**(빈칸 채우기) &nbsp;&nbsp;[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/baekja/Lecture_DL-AE/blob/main/PyTorch_MNIST_CNN/mnist_cnn_complete.ipynb) ← **완성본**(정답·실행결과 포함)

> 이론 시간에 강의노트 그림으로 따라가 본 **합성곱 · 스트라이드 · 패딩 · 풀링** 계산을 기억하나요?
> 오늘은 그 그림의 숫자를 그대로 PyTorch에 넣어 **똑같은 결과**가 나오는지 하나씩 확인합니다.
> 그다음 이 부품들로 합성곱 신경망(CNN)을 조립해, 손글씨 숫자 시험 1만 장에서 **98.5 % 이상** 맞혀 봅시다.

## 📌 강의 중점
- 강의노트의 합성곱 · 스트라이드 · 패딩 · 풀링 · 평탄화 그림을 PyTorch로 **그대로 재현**
- 강의 실습의 CNN(합성곱 → ReLU → 풀링 × 2 → 평탄화 → 완전연결)을 `nn.Module` 로 구현
- 층마다 **모양(shape)과 파라미터 수**를 공식으로 예측하고 코드로 확인 · 학습된 필터와 특징 맵 들여다보기

## 🎯 학습 목표
1. `F.conv2d` · `F.max_pool2d` · `F.avg_pool2d` 로 강의노트 그림의 숫자를 재현할 수 있다.
2. 출력 크기 공식 $(N - F + 2P)/S + 1$ 로 층마다 출력 모양을 예측하고 코드로 확인할 수 있다.
3. `nn.Conv2d` · `nn.MaxPool2d` · `nn.Linear` 로 CNN을 만들고 층별 파라미터 수를 공식으로 계산할 수 있다.
4. CNN을 학습시켜 MNIST 시험 정확도 98.5 % 이상을 얻고, 필터 · 특징 맵 · 틀린 예를 해석할 수 있다.

## 📋 이 노트북 사용법

| 표시 | 뜻 |
|:-:|---|
| ▶ | 셀 실행 — 셀을 클릭하고 `Shift + Enter` |
| 🎬 EP*n* | 이 부분을 설명하는 강의 영상 에피소드 (e-campus) |
| ⏸ | 영상을 멈추고 직접 실행해 보는 지점 |
| ✅ / ❌ | 자가검증 결과 — ❌ 가 나오면 힌트를 읽고 고쳐 보세요 |
| 🔎 | 강의노트 그림과 코드 결과를 견주어 보는 '차이 찾기' |
| ✏️ | 빈칸 `____` 을 채워야 하는 줄 (주석 끝의 힌트 참고) |

**주석(`#` 뒤의 글) 읽는 법** — 실행에는 영향이 없는 설명입니다.
- `①②③`, `1️⃣2️⃣3️⃣4️⃣` 순서(단계) · `💡` 이론 연결(합성곱 신경망 강의) · `📐` 데이터 모양(shape) · `⚠️` 흔한 실수 · `✅` 기대 결과 · `⏱` 걸린 시간

**Colab에서 처음 열었다면**
1. 메뉴 **파일 → Drive에 사본 저장** 을 먼저 누르세요. (사본에서 작업해야 내 Drive에 저장됩니다)
2. 5장에 학습이 있으니 **GPU 런타임을 추천**합니다: 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU**. CPU 그대로 두어도 모든 셀이 실행됩니다(5장 학습이 4~5분 걸림).
3. 위에서부터 **차례대로** 실행하세요. 런타임이 초기화되어 변수가 사라지면 처음부터 다시 실행하면 됩니다.
4. ✏️ 표시된 줄의 `____` 을 채운 뒤 실행하세요. **'모두 실행'은 첫 빈칸에서 멈춥니다** — 하나씩 채우며 내려가세요.

**🎬 영상과 함께 보는 법 (반복 학습 추천 순서)**
1회차 영상 전체 보기 → 2회차 실습본을 열고 ⏸ 마다 멈추며 따라 치기 → 3회차 영상 없이 빈칸 채우기 → 모든 ✅ 통과 → 5장 도전 과제

---
## 0. 준비 — 라이브러리 불러오기  🎬 EP1

이 셀에서 할 일: PyTorch와 torchvision(이미지 데이터 도구)을 불러오고, 계산할 장치(CPU 또는 GPU)를 정합니다.

> 💡 **다시 보는 파이썬 문법 — `import … as 별명`, `from … import …`**
>
> `import torch.nn.functional as F` 는 긴 이름에 짧은 별명 `F` 를 붙입니다. 이제 `F.conv2d(...)` 처럼 씁니다.
> `from torchvision import datasets` 는 도구 상자에서 `datasets` 하나만 꺼내 옵니다.

In [ ]:
import torch                                   # PyTorch: 딥러닝 도구 상자
from torch import nn                           # nn = 신경망 부품(층) 모음
import torch.nn.functional as F                # F = 합성곱 · 풀링을 '함수'로 바로 쓰는 모음
from torch.utils.data import DataLoader        # 데이터를 미니배치로 나눠 주는 도구
from torchvision import datasets               # 이미지 데이터셋 모음 (MNIST, FashionMNIST …)
from torchvision.transforms import ToTensor    # 그림 → 텐서(0~1 실수) 변환
import matplotlib.pyplot as plt                # 그래프 도구 (별명 plt)
import time                                    # 걸린 시간 재기 (⏱)

torch.set_printoptions(precision=4, sci_mode=False)      # 소수 4자리, 지수 표기 끄기
device = "cuda" if torch.cuda.is_available() else "cpu"  # GPU가 있으면 GPU, 없으면 CPU
print("PyTorch 버전:", torch.__version__)                # Colab에는 이미 설치됨
print("계산 장치:", device)                              # cuda = GPU, cpu = CPU

### 자가검증 도우미 `check()` · `check_range()` 준비
아래 셀은 결과가 맞는지 ✅ / ❌ 로 알려 주는 **도우미 함수**입니다.
- `check()` : 지난 실습과 같습니다. 기대값과 허용오차(`tol`) 이내로 같으면 ✅ (`tol=0` 이면 완전히 같아야 ✅)
- `check_range()` : 정확도처럼 **범위 안에 들어오면** ✅ (컴퓨터 · GPU 마다 조금씩 달라지는 값에 씁니다)

지금은 내용을 몰라도 괜찮습니다 — **실행만** 해 두세요.

In [ ]:
def check(name, value, expected, tol=0.01):  # 자가검증 (이름, 결과, 기대값, 허용오차)
    """value 가 expected 와 tol 이내로 같으면 ✅, 다르면 ❌ 를 보여 줍니다."""
    value = torch.as_tensor(value, dtype=torch.float32)        # 숫자·리스트 → 텐서
    expected = torch.as_tensor(expected, dtype=torch.float32)  # 기대값도 텐서로
    if value.shape != expected.shape:                          # ⚠️ 모양부터 다르면
        print(f"❌ {name}: 모양 {tuple(value.shape)} ≠ 기대 {tuple(expected.shape)}")
        return                                                 # 여기서 끝
    gap = (value - expected).abs().max().item()                # 가장 큰 차이
    mark = "✅" if gap <= tol else "❌"                         # 허용오차 안이면 통과
    tol_s = f"{tol:.6f}".rstrip("0").rstrip(".")               # 0.000001 처럼 보기 좋게
    print(f"{mark} {name}: 최대 차이 {gap:.6f} (허용 {tol_s})")  # 결과 출력

def check_range(name, value, lo, hi):        # 자가검증 (이름, 결과, 최솟값, 최댓값)
    """value 가 lo ~ hi 범위 안이면 ✅, 벗어나면 ❌ 를 보여 줍니다."""
    value = float(value)                                       # 텐서·숫자 → 파이썬 실수
    mark = "✅" if lo <= value <= hi else "❌"                  # 범위 안이면 통과
    print(f"{mark} {name}: {value:.4f} (기대 범위 {lo} ~ {hi})")  # 결과 출력

---
## 1. MNIST — 손글씨 숫자 7만 장  🎬 EP1

MNIST 는 0~9 손글씨 숫자 그림 모음입니다. 한 장은 **28×28 흑백**이고, 학습용 6만 장 + 시험용 1만 장입니다.
지난 실습(FashionMNIST)과 모양이 똑같아서, 데이터 이름만 `datasets.MNIST` 로 바꾸면 됩니다.

- `root="data"` : 내려받을 폴더 · `train=True / False` : 학습용 / 시험용 · `download=True` : 없으면 인터넷에서 받기
- `transform=ToTensor()` : 그림(0~255 정수)을 텐서(0~1 실수)로 바꾸기

⏸ 처음 실행하면 내려받느라 10~30초쯤 걸립니다.

In [ ]:
train_data = datasets.MNIST(root="data", train=True, download=True,  # 학습용 6만 장
                            transform=________())                     # ✏️ 그림 → 텐서 (힌트: To…)
test_data = datasets.MNIST(root="data", train=False, download=True,  # 시험용 1만 장
                           transform=________())                      # ✏️ 같은 변환 (힌트: 위 줄)
print("학습 데이터:", len(train_data), "장 / 시험 데이터:", len(test_data), "장")  # 6만 / 1만
img, label = train_data[0]                         # 첫 번째 샘플 = (그림, 정답 숫자)
print("그림 모양:", img.shape)                      # 📐 [1, 28, 28] = [채널, 높이, 너비]
print("정답(label):", label)                       # 이 그림이 나타내는 숫자
print("픽셀 값 범위:", img.min().item(), "~", img.max().item())  # ToTensor → 0 ~ 1
check("그림 모양", img.shape, (1, 28, 28), tol=0)    # ✅ 흑백 = 채널 1개

🔍 **결과 읽는 법** — `[1, 28, 28]` 은 **[채널, 높이, 너비]** 입니다. 흑백이라 채널이 1개이고, 컬러 사진이면 3개(R, G, B)입니다(강의노트 '이미지 데이터의 구조'). 픽셀은 0(검정) ~ 1(흰색) 사이의 실수입니다.

### 1-1. 손글씨 구경하기
학습 데이터 앞 16장입니다. (그림을 그리는 코드입니다 — 결과 그림만 보셔도 됩니다)

> 💡 **새 파이썬 문법 — `enumerate`**
>
> `for i, ax in enumerate(목록):` 은 번호 `i` (0, 1, 2, …)와 값 `ax` 를 **함께** 꺼내 줍니다.

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(12, 3.4))   # 그림 칸 2줄 × 8칸 = 16장
for i, ax in enumerate(axes.flat):                   # i = 0, 1, …, 15 와 그림 칸을 함께
    img, label = train_data[i]                       # i번째 학습 샘플
    ax.imshow(img[0], cmap="gray")                   # img[0]: 채널을 빼고 28×28 만 그리기
    ax.set_title(f"label {label}")                   # 정답 숫자 (영문: Colab 한글 깨짐 방지)
    ax.axis("off")                                   # 눈금 감추기
plt.tight_layout()                                   # 간격 정리
plt.show()                                           # 그리기

🔍 같은 숫자라도 사람마다 모양이 다르죠. 숫자를 알아보는 단서는 픽셀 하나가 아니라 **획 · 모서리 · 곡선처럼 이웃한 픽셀들이 만드는 모양**입니다. CNN은 바로 이 '이웃의 모양'을 보는 신경망입니다.

### 1-2. DataLoader — 64장씩 미니배치로
학습할 때는 6만 장을 한꺼번에 넣지 않고 **64장씩 묶어(미니배치)** 넣습니다(지난 실습과 같음).
학습용은 에포크마다 순서를 섞되(`shuffle=True`), `generator` 로 섞는 '씨앗'을 고정해 언제 실행해도 같은 순서가 되게 합니다(재현용).

In [ ]:
BATCH = 64                                                             # 미니배치 크기 (강의 실습)
train_loader = DataLoader(train_data, batch_size=BATCH, shuffle=True,  # 학습용: 섞어서
                          generator=torch.Generator().manual_seed(0))  # 섞는 순서 고정
test_loader = DataLoader(test_data, batch_size=BATCH, shuffle=False)   # 시험용: 순서 그대로
X, y = next(iter(train_loader))                     # 첫 번째 미니배치 하나 꺼내 보기
print("X 모양 [N, C, H, W]:", X.shape)               # 📐 [64, 1, 28, 28]
print("y 모양:", y.shape, "→ 앞 10개 정답:", y[:10].tolist())  # 📐 [64] = 정답 숫자 64개
print("미니배치 수: 학습", len(train_loader), "/ 시험", len(test_loader))  # 938 / 157
check("미니배치 X 모양", X.shape, (64, 1, 28, 28), tol=0)  # ✅ N=64, C=1, H=W=28

🔍 **결과 읽는 법** — `[64, 1, 28, 28]` 은 **[N, C, H, W]** = [샘플 수, 채널, 높이, 너비] 입니다. PyTorch의 합성곱 층에는 이 **4차원** 모양으로 넣는 것이 기본입니다. 6만 ÷ 64 = 937.5 라서 학습용 미니배치는 938개(마지막 묶음은 32장)입니다.

### 1-3. 왜 CNN인가? — 모든 픽셀을 모든 노드에 연결하면
지난 실습의 다층 신경망(MLP)은 그림을 784개 숫자로 펼쳐, 첫 은닉층 512개 노드에 **모두** 연결했습니다.
강의노트의 예처럼 **400×700 컬러 사진**이라면 어떻게 될까요? 합성곱 층과 가중치 개수를 비교해 봅시다.

> 💡 `.numel()` = 텐서 안의 숫자(원소) 개수 · f-string 의 `{값:,}` = 천 단위 쉼표 (401,408 처럼)

In [ ]:
n_mnist = 28 * 28 * 1                      # MNIST 한 장의 입력 수 (흑백)
n_photo = 400 * 700 * 3                    # 💡 강의노트 예: 400×700 컬러(RGB) 사진 = N1
print(f"MNIST 784개 → 은닉 512개: 가중치 {n_mnist * 512:,}개")   # 401,408
print(f"사진 {n_photo:,}개 → 은닉 512개: 가중치 {n_photo * 512:,}개")  # 4억 3천만 😱
conv = nn.Conv2d(3, 32, kernel_size=3)     # 비교: 3×3 필터 32개짜리 합성곱 층 (컬러 입력 3채널)
n_conv = conv.weight.numel() + conv.bias.numel()  # 가중치 (32, 3, 3, 3) = 864개 + 편향 32개
print(f"합성곱 층(3×3 필터 32개): {n_conv:,}개 — 사진 크기와 상관없음")  # 896
check("입력 수 400×700×3 (강의노트 N1)", n_photo, 840000, tol=0)     # ✅
check("합성곱 층 파라미터 (3×3×3 + 1)×32", n_conv, 896, tol=0)       # ✅

🔍 **결과 읽는 법** — 사진을 은닉노드 512개에 모두 연결하면 가중치가 **4억 3천만 개**입니다(강의노트: "학습이 어렵다!"). 합성곱 층은 3×3 필터 32개라 896개뿐이고, 사진이 커져도 늘지 않습니다.

| 이미지의 특징 (강의노트) | 다층 신경망(MLP) | CNN |
|---|---|---|
| 이웃 픽셀끼리 관계가 깊다 (spatially-local correlation) | 모든 픽셀을 모든 노드에 연결 | **이웃(3×3)만 연결** (sparse connection) |
| 같은 모양이 어느 위치에나 나타난다 (invariant feature) | 위치마다 다른 가중치 | **같은 필터를 모든 위치에** (shared weight) |
| 2차원 구조 | 처음부터 한 줄로 펼침 | 2차원 그대로 계산 |

→ 이 두 아이디어(이웃만 · 같은 필터)를 실현한 계산이 바로 **합성곱(convolution)** 입니다. 2장에서 강의노트 그림의 숫자로 직접 해 봅니다.

**📝 1분 퀴즈** ① 미니배치 `X` 의 모양 `[64, 1, 28, 28]` 에서 `1` 은 무엇일까요? ② 합성곱 층의 파라미터 수가 사진 크기와 상관없는 이유는?
<details><summary>정답 보기</summary>

① **채널 수** — 흑백이라 1 (컬러 사진이면 3)
② 작은 필터(3×3) 하나를 그림의 **모든 위치에서 같이 쓰기(shared weight)** 때문입니다. 가중치는 '필터 크기 × 개수'만큼만 있습니다.
</details>

---
## 2. 합성곱(Convolution) — 강의노트 그림을 PyTorch로  🎬 EP2

강의노트 그림의 5×5 입력과 3×3 필터(커널)입니다. 필터를 입력의 왼쪽 위에 겹쳐 놓고, **같은 자리끼리 곱해서 모두 더하면** 특징 맵(feature map)의 첫 칸이 됩니다. 그림에서는 **4** 였습니다.

```
입력 X5 (5×5)        필터 K (3×3)
1  1  1  0  0        1  0  1
0  1  1  1  0        0  1  0
0  0  1  1  1        1  0  1
0  0  1  1  0
0  1  1  0  0
```

> 💡 **새 파이썬 문법 — 2차원 슬라이싱 `X5[0:3, 0:3]`**
>
> `행 범위, 열 범위` 를 함께 씁니다. `0:3` 은 0, 1, 2 (끝 번호 3은 빠짐) → 왼쪽 위 3×3 조각입니다.

In [ ]:
X5 = torch.tensor([[1., 1., 1., 0., 0.],   # 강의노트 그림의 5×5 입력 — 1행
                   [0., 1., 1., 1., 0.],   # 2행
                   [0., 0., 1., 1., 1.],   # 3행
                   [0., 0., 1., 1., 0.],   # 4행
                   [0., 1., 1., 0., 0.]])  # 5행
K = torch.tensor([[1., 0., 1.],            # 강의노트 그림의 3×3 필터 (초록 칸의 ×1, ×0)
                  [0., 1., 0.],            # 2행
                  [1., 0., 1.]])           # 3행
patch = X5[0:3, 0:3]                       # 왼쪽 위 3×3 조각 (행 0~2, 열 0~2)
first = (patch * K).sum()                  # ① 같은 자리끼리 곱하고 ② 모두 더하기
print("왼쪽 위 3×3 조각:\n", patch)          # 필터와 겹친 부분
print("특징 맵 첫 칸 =", first.item())       # 그림: 4
check("특징 맵 첫 칸 (그림: 4)", first, 4, tol=0)  # ✅ 그림의 빨간 칸

### 2-1. 필터를 한 칸씩 밀며(stride 1) 특징 맵 전체 만들기
필터를 오른쪽으로 한 칸, 다 가면 아래로 한 칸 … 이렇게 밀면 5×5 입력에서 **3×3 특징 맵**이 나옵니다(강의노트 stride 1 그림).
먼저 `for` 문으로 직접 계산해 봅시다.

> 💡 **새 파이썬 문법 — 이중 `for` 문**
>
> `for` 안에 `for` 를 넣으면 (i, j) = (0,0), (0,1), (0,2), (1,0), … (2,2) 처럼 **표의 모든 칸**을 차례로 돕니다.

In [ ]:
fmap_hand = torch.zeros(3, 3)                    # 손계산 결과를 담을 3×3 빈 표
for i in range(3):                               # i = 특징 맵의 행 (0, 1, 2)
    for j in range(3):                           # j = 특징 맵의 열 (0, 1, 2)
        piece = X5[i:i+3, j:j+3]                 # (i, j) 에서 시작하는 3×3 조각
        fmap_hand[i, j] = (piece * K).sum()      # 곱해서 모두 더하기
print("for 문으로 계산한 특징 맵:\n", fmap_hand)   # 📐 3×3

### 2-2. PyTorch 한 줄 — `F.conv2d`
PyTorch의 `F.conv2d(입력, 필터)` 가 위의 이중 `for` 문을 한 번에 해 줍니다.
PyTorch 합성곱은 입력을 `[N, C, H, W]`, 필터를 `[필터 수, 채널, 높이, 너비]` 의 **4차원**으로 주고받는 것이 기본 약속입니다. 그래서 `.view(1, 1, 5, 5)` 로 모양만 바꿉니다(숫자는 그대로).

> 💡 수학의 '합성곱'은 필터를 180° 뒤집어서 곱하지만, 딥러닝에서는 뒤집지 않고 **그대로 겹쳐서** 곱합니다. 강의노트 그림과 PyTorch 모두 이 방식입니다. (이 필터는 뒤집어도 모양이 같아서 어느 쪽이든 결과가 같습니다)

In [ ]:
x5 = X5.view(1, 1, 5, 5)                   # 📐 입력 [N, C, H, W] = [1장, 채널 1, 5, 5]
k3 = K.view(1, 1, 3, 3)                    # 📐 필터 [필터 1개, 채널 1, 3, 3]
fmap = F.conv2d(______)                    # ✏️ 합성곱 한 줄!  📐 → [1, 1, 3, 3] (힌트: 위 두 줄)
print("F.conv2d 특징 맵:\n", fmap[0, 0])     # [0, 0] = 첫 장의 첫 채널
check("F.conv2d = for 문 손계산", fmap[0, 0], fmap_hand, tol=0)       # ✅ 9칸 모두 같음
check("특징 맵 첫 칸 (그림: 4)", fmap[0, 0, 0, 0], 4, tol=0)           # ✅

🔍 **결과 읽는 법** — 그림에는 첫 칸(4)만 적혀 있었죠. 나머지 여덟 칸까지 아홉 칸 전체 `[[4, 3, 4], [2, 4, 3], [2, 3, 4]]` 가 `for` 문 손계산과 PyTorch에서 **완전히 같습니다(차이 0)**. 공책에 한 칸만 더 계산해 보세요 — 예를 들어 첫 행 둘째 칸(3)은 필터를 오른쪽으로 한 칸 민 자리입니다.

### 2-3. 스트라이드(stride) 2 — 두 칸씩 건너뛰기
강의노트의 stride 2 그림(5×5 입력, 3×3 필터 → **2×2** 특징 맵)을 같은 숫자로 계산해 봅시다.

> 💡 **새 파이썬 문법 — `[::2]` 간격 꺼내기**
>
> `a[::2]` 는 처음부터 끝까지 **두 칸 간격**으로 꺼냅니다. `fmap[0, 0, ::2, ::2]` = 행도 열도 한 칸씩 건너뛴 값.

In [ ]:
fmap_s2 = F.conv2d(x5, k3, ________)         # ✏️ 필터를 두 칸씩 이동 (힌트: 이동 간격 옵션)
print("stride 2 특징 맵:\n", fmap_s2[0, 0])    # 📐 2×2
check("stride 2 모양 (그림: 2×2)", fmap_s2[0, 0].shape, (2, 2), tol=0)        # ✅
check("stride 2 = stride 1 결과의 한 칸 건너 값", fmap_s2[0, 0], fmap[0, 0, ::2, ::2], tol=0)  # ✅

🔍 stride 2 는 stride 1 특징 맵에서 **한 칸씩 건너뛴 자리만** 계산한 것과 같습니다(4, 4, 2, 4). 계산은 줄지만 그만큼 놓치는 정보도 생깁니다 — 강의노트: "Large stride → 특징을 놓칠 가능성 증가".

### 2-4. 출력 크기 공식 — 계산하기 전에 미리 알 수 있다
**출력 크기** $= \dfrac{N - F + 2P}{S} + 1$  (N: 입력 크기 · F: 필터 크기 · P: 패딩 · S: 스트라이드)

> 💡 **새 파이썬 문법 — `//` (나눗셈의 몫)**
>
> `7 // 2` = 3 처럼 소수점 아래를 버린 몫입니다. 나누어떨어지지 않을 때 PyTorch도 소수점 아래를 버립니다.

> 💡 강의노트(실습)의 주석 `(input_size - kernel_size) / stride + 1` 은 패딩이 없을 때의 식입니다. 패딩이 있으면 그 바로 아래 예시처럼 `2P` 를 더합니다.

In [ ]:
def out_size(n, f, p, s):                     # 출력 크기 (입력 n, 필터 f, 패딩 p, 스트라이드 s)
    return (n - f + 2 * p) // s + 1           # 💡 (N − F + 2P) / S + 1  (// = 몫)

print("5×5, 필터 3, stride 1 →", out_size(5, 3, 0, 1))    # 2-2 의 특징 맵 크기
print("5×5, 필터 3, stride 2 →", out_size(5, 3, 0, 2))    # 2-3 의 특징 맵 크기
check("공식 = stride 1 실제 크기", out_size(5, 3, 0, 1), fmap.shape[-1], tol=0)    # ✅ 3
check("공식 = stride 2 실제 크기", out_size(5, 3, 0, 2), fmap_s2.shape[-1], tol=0)  # ✅ 2

### 2-5. 제로 패딩(zero padding) — 크기를 지키는 0 테두리
합성곱을 할수록 특징 맵이 작아지고, 가장자리 픽셀은 계산에 덜 쓰입니다. 그래서 테두리에 **0 을 한 칸** 둘러 줍니다(강의노트 그림: 6×6 입력 → padding size 1 → **8×8**).

```
3  1  1  2  8  4
1  0  7  3  2  6
2  3  5  1  1  3
1  4  1  2  6  5
3  2  1  3  7  2
9  2  6  2  5  1
```

In [ ]:
X6 = torch.tensor([[3., 1., 1., 2., 8., 4.],   # 강의노트 그림의 6×6 입력 — 1행
                   [1., 0., 7., 3., 2., 6.],   # 2행
                   [2., 3., 5., 1., 1., 3.],   # 3행
                   [1., 4., 1., 2., 6., 5.],   # 4행
                   [3., 2., 1., 3., 7., 2.],   # 5행
                   [9., 2., 6., 2., 5., 1.]])  # 6행
X6p = F.pad(X6, (1, 1, 1, 1))                 # 테두리에 0 을 한 칸씩 (왼, 오, 위, 아래)
print(X6p)                                    # 📐 8×8 — 그림 오른쪽과 같은지 보세요
check("패딩 후 모양 (그림: 8×8)", X6p.shape, (8, 8), tol=0)       # ✅
check("가운데 6×6 = 원래 입력", X6p[1:7, 1:7], X6, tol=0)          # ✅ 숫자는 그대로
check("테두리 칸의 합 = 0", X6p.sum() - X6.sum(), 0, tol=0)         # ✅ 테두리는 모두 0

In [ ]:
x6 = X6.view(1, 1, 6, 6)                         # 📐 [N, C, H, W] 모양으로
no_pad = F.conv2d(x6, k3)                        # 패딩 없이 (앞에서 만든 3×3 필터 k3)
with_pad = F.conv2d(x6, k3, padding=1)           # 패딩 1 — 합성곱 안에서 0 테두리
print("패딩 없음:", tuple(no_pad.shape), "/ 패딩 1:", tuple(with_pad.shape))  # 6 → 4 / 6 → 6
check("공식 (6 − 3 + 0)/1 + 1 = 4", out_size(6, 3, 0, 1), no_pad.shape[-1], tol=0)   # ✅
check("공식 (6 − 3 + 2)/1 + 1 = 6", out_size(6, 3, 1, 1), with_pad.shape[-1], tol=0)  # ✅
check("padding=1 = F.pad 후 합성곱", with_pad, F.conv2d(X6p.view(1, 1, 8, 8), k3), tol=0)  # ✅

### 2-6. 진짜 층 `nn.Conv2d` 에도 공식이 맞을까?
CNN 안에서는 `F.conv2d` 대신, 필터를 **학습할 가중치로 품고 있는 층** `nn.Conv2d(입력 채널, 필터 수, kernel_size, stride, padding)` 을 씁니다.
1장에서 꺼낸 미니배치 `X` (그림 64장)를 강의 실습의 첫 합성곱 층에 통과시켜 봅시다. 필터가 32개이므로 특징 맵도 **32장**이 나옵니다.

In [ ]:
conv_a = nn.Conv2d(1, 32, kernel_size=3, _________)  # ✏️ 강의 실습의 첫 합성곱 층 (패딩 1)
conv_b = nn.Conv2d(1, 32, kernel_size=3)             # 비교: 패딩 없음 (padding=0)
print("입력      :", tuple(X.shape))                  # 📐 (64, 1, 28, 28)
print("패딩 1    :", tuple(conv_a(X).shape))          # 📐 (64, 32, 28, 28) — 크기 유지
print("패딩 없음 :", tuple(conv_b(X).shape))          # 📐 (64, 32, 26, 26) — 2 줄어듦
check("공식 = 실제 층 (패딩 1)", out_size(28, 3, 1, 1), conv_a(X).shape[-1], tol=0)   # ✅ 28
check("공식 = 실제 층 (패딩 없음)", out_size(28, 3, 0, 1), conv_b(X).shape[-1], tol=0)  # ✅ 26

🔍 **결과 읽는 법** — 채널 수는 1 → **32** (필터 32개 = 특징 맵 32장), 크기는 공식대로 28 (패딩 1) 또는 26 (패딩 없음)입니다. 강의 실습의 CNN이 `padding=1` 을 쓰는 이유가 바로 '크기 유지'입니다.

**📝 1분 퀴즈** ① 7×7 입력에 3×3 필터, stride 2, 패딩 0 이면 출력 크기는? ② 28×28 입력에 5×5 필터(stride 1)를 쓰면서 크기 28 을 지키려면 패딩은?
<details><summary>정답 보기</summary>

① (7 − 3 + 0) / 2 + 1 = **3** → 3×3
② (28 − 5 + 2P) / 1 + 1 = 28 → **P = 2**
</details>

---
## 3. 풀링(Pooling) — 특징 맵을 절반으로 요약  🎬 EP3

풀링은 특징 맵을 2×2 창(window)으로 나누어 **한 숫자로 요약**합니다(강의노트 그림: 4×4 → 2×2).
- **최댓값 풀링(max pooling)** : 창에서 가장 큰 값 · **평균값 풀링(average pooling)** : 창의 평균
- 학습할 가중치가 **없는** 층이고, 창 크기와 스트라이드를 보통 같게(2) 둡니다.

```
28  80 | 35  97
12  67 | 73  86
-------+-------
25  25 | 56  72
25  25 | 84  39
```

In [ ]:
X4 = torch.tensor([[28., 80., 35., 97.],      # 강의노트 그림의 4×4 특징 맵 — 1행
                   [12., 67., 73., 86.],      # 2행
                   [25., 25., 56., 72.],      # 3행
                   [25., 25., 84., 39.]])     # 4행
x4 = X4.view(1, 1, 4, 4)                      # 📐 [N, C, H, W] 모양으로
mx = F.__________(x4, kernel_size=2)          # ✏️ 2×2 창마다 최댓값 (stride 도 2) (힌트: max 풀링)
print("최댓값 풀링:\n", mx[0, 0])               # 📐 2×2
check("최댓값 풀링 (그림 a: 80, 97, 25, 84)", mx[0, 0], [[80, 97], [25, 84]], tol=0)  # ✅

### 3-1. 평균값 풀링 — ✋ 먼저 예측해 보세요
그림 (b) 의 평균값 풀링 결과는 **46, 72, 25, 62** 입니다. PyTorch도 똑같이 나올까요?
(가) 똑같이 나온다 (나) 조금 다르게 나온다

In [ ]:
av = F.avg_pool2d(x4, kernel_size=2)          # 2×2 창마다 평균
print("평균값 풀링 (PyTorch):\n", av[0, 0])     # 소수점이 보이나요?
print("그림 (b): [[46, 72], [25, 62]]")        # 강의노트 그림의 값
print("왼쪽 위 창 손계산:", (28 + 80 + 12 + 67) / 4)  # 187 ÷ 4

### 3-2. 🔎 그림 속 차이 찾기
PyTorch는 **46.75**, 그림은 **46** — 손으로 계산해도 187 ÷ 4 = 46.75 입니다. 계산 방법(2×2 창의 평균)은 같은데, 그림의 숫자는 어떻게 나온 걸까요?
✋ (가) 소수점 아래를 **반올림** (나) 소수점 아래를 **버림**. 코드로 확인해 봅시다.

In [ ]:
print("반올림:", av[0, 0].round())            # .round(): 반올림
print("버림  :", av[0, 0].floor())            # .floor(): 소수점 아래 버림 (내림)
check("그림 (b) = 평균의 소수점 아래 버림", av[0, 0].floor(), [[46, 72], [25, 62]], tol=0)  # ✅
check("평균값 풀링 (손계산 46.75, 72.75 …)", av[0, 0], [[46.75, 72.75], [25, 62.75]], tol=0)  # ✅

🔍 정답은 **(나)** — 그림 (b) 는 칸에 정수만 적느라 **소수점 아래를 버린 값**입니다(46.75 → 46). 반올림이었다면 47, 73, 25, 63 이었겠죠. 계산 방법은 그림과 PyTorch가 똑같고, **표시 방법만** 다릅니다. 이렇게 그림의 숫자를 코드로 직접 확인해 보는 습관이 중요합니다. 👏

> 💡 이미지 분석에서는 주로 **최댓값 풀링**을 씁니다(강의노트). 창 안에서 가장 강하게 나타난 특징을 그대로 남기기 때문입니다.

### 3-3. 합성곱 → ReLU → 풀링, 진짜 손글씨로
손으로 만든 필터로 MNIST 숫자 하나를 통과시켜 봅시다.
- **세로 경계 필터** (왼쪽 열 −1, 오른쪽 열 +1) : '왼쪽은 어둡고 오른쪽은 밝은' 경계에서 큰 양수, 반대 경계에서 음수
- **ReLU** = max(0, x) : 음수를 0 으로 지우는 활성화 함수(강의노트: 합성곱 결과의 비선형 변환)
- **최댓값 풀링** : 크기를 절반으로

In [ ]:
digit, digit_label = test_data[0]              # 시험 데이터 첫 장 (숫자 7)
edge = torch.tensor([[-1., 0., 1.],            # 세로 경계 필터: 왼쪽 열 −1 · 가운데 0 · 오른쪽 +1
                     [-1., 0., 1.],            # 2행 (1행과 같음)
                     [-1., 0., 1.]])           # 3행 (1행과 같음)
d = digit.view(1, 1, 28, 28)                   # 📐 [N, C, H, W] = 1장짜리 미니배치
z = F.conv2d(d, edge.view(1, 1, 3, 3), padding=1)  # ① 합성곱 (패딩 1 → 28×28 유지)
a = F.relu(z)                                  # ② ReLU: 음수 → 0  💡 max(0, x)
p = F.max_pool2d(a, kernel_size=2)             # ③ 최댓값 풀링: 28 → 14
print("정답:", digit_label, "| 모양:", tuple(z.shape), "→", tuple(p.shape))  # 📐 28 → 14
print(f"합성곱 결과 범위: {z.min():.1f} ~ {z.max():.1f}")      # 음수(−)와 양수(+)가 섞여 있음
print(f"ReLU 뒤 범위    : {a.min():.1f} ~ {a.max():.1f}")      # 음수가 사라짐
check("ReLU 후 최솟값 = 0", a.min(), 0, tol=0)          # ✅ 음수 없음
check("풀링 후 모양", p.shape, (1, 1, 14, 14), tol=0)   # ✅ 크기 절반

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(12, 3.2))      # 그림 4개를 가로로
steps = [(digit[0], "input 28x28"), (z[0, 0], "conv (edge filter)"),  # (그림, 제목) 목록
         (a[0, 0], "ReLU"), (p[0, 0], "max pool 14x14")]              # 단계 순서대로
for ax, (im, title) in zip(axes, steps):               # 칸과 (그림, 제목)을 짝지어
    ax.imshow(im, cmap="gray")                         # 흑백으로 그리기 (밝을수록 큰 값)
    ax.set_title(title)                                # 제목 (영문)
    ax.axis("off")                                     # 눈금 감추기
plt.tight_layout()                                     # 간격 정리
plt.show()                                             # 그리기

🔍 **결과 읽는 법** — 합성곱 그림에서 숫자 7 의 획 **왼쪽 경계는 밝게(+), 오른쪽 경계는 어둡게(−)** 나타납니다. ReLU가 음수를 0 으로 지워 한쪽 경계만 남기고, 풀링은 그 특징을 지닌 채 크기를 14×14 로 줄입니다.
CNN은 이런 필터 32개, 64개를 사람이 정해 주지 않아도 **학습으로 스스로 찾아냅니다** — 5장에서 학습된 필터를 들여다봅니다.

### 3-4. 평탄화(Flatten) — 3차원 특징 맵 → 1차원 한 줄
특징 추출이 끝나면 완전연결층에 넣기 위해 특징 맵을 **한 줄로 펼칩니다**. 강의노트 그림은 2×2 특징 맵 3장(파랑 · 초록 · 빨강)을 숫자 12개짜리 막대로 펼쳤습니다.
PyTorch는 **채널 순서대로**, 채널 안에서는 **행 순서대로** 펼칩니다.

In [ ]:
maps = torch.tensor([[[1., 2.], [1., 1.]],     # 강의노트 그림: 파랑 특징 맵 (2×2)
                     [[1., 3.], [3., 1.]],     # 초록 특징 맵
                     [[4., 2.], [1., 2.]]])    # 빨강 특징 맵   📐 (3, 2, 2)
flat = maps.view(-1)                           # 전부 한 줄로 (-1 = 개수는 알아서 계산)
print("평탄화:", flat)                          # 📐 12개 = 3 × 2 × 2
check("평탄화 (그림의 세로 막대)", flat, [1, 2, 1, 1, 1, 3, 3, 1, 4, 2, 1, 2], tol=0)  # ✅

### 3-5. 미니배치를 평탄화할 때 — 샘플 수는 지키기
CNN에서는 64장을 한꺼번에 평탄화합니다. 이때 **첫 번째 축(샘플 수 64)은 그대로** 두고, 나머지 (채널 × 높이 × 너비)만 펼쳐야 합니다.
`x.view(x.size(0), -1)` — `x.size(0)` 은 첫 번째 축의 크기(64), `-1` 은 "나머지는 알아서 계산".

In [ ]:
fm = torch.zeros(64, 64, 7, 7)                 # 예: 두 번째 풀링 뒤의 모양 [N, C, H, W]
good = fm.view(__________, -1)                 # ✏️ 샘플마다 한 줄씩: (64, 64×7×7) (힌트: 첫 축 크기)
bad = fm.view(-1)                              # ⚠️ 64장이 통째로 한 줄에 섞임
print("올바른 평탄화:", tuple(good.shape))        # 📐 (64, 3136)
print("잘못된 평탄화:", tuple(bad.shape))         # 📐 (200704,) — 샘플 구분이 사라짐
check("평탄화 모양 (64, 64 × 7 × 7)", good.shape, (64, 3136), tol=0)  # ✅ 3136

### 3-6. 블록 하나 = 합성곱 → ReLU → 풀링
강의노트의 CNN 핵심 구성요소 **Convolution → ReLU → Pooling** 을 `nn` 부품으로 이어 붙이면 '블록' 하나가 됩니다. 이번에는 필터를 손으로 정하지 않고, `nn.Conv2d` 가 필터 32개를 **학습할 가중치**로 품습니다.

> 💡 `nn.Sequential(층1, 층2, …)` : 층을 순서대로 이어 붙이는 간단한 설계도입니다(지난 실습에서 봄).

In [ ]:
block = nn.Sequential(nn.Conv2d(1, 32, kernel_size=3, padding=1),   # 합성곱: 필터 32개 (학습 대상)
                      nn.ReLU(),                                     # 활성화: max(0, x)
                      nn.MaxPool2d(kernel_size=2, stride=2))         # 풀링: 크기 절반
out = block(X)                                 # 1장에서 꺼낸 미니배치(그림 64장)를 통과
print("블록 통과:", tuple(X.shape), "→", tuple(out.shape))   # 📐 → (64, 32, 14, 14)
check("블록 1 출력 모양", out.shape, (64, 32, 14, 14), tol=0)  # ✅ 채널 32 · 크기 절반

🔍 블록 하나를 지나면 **채널은 늘고(1 → 32), 크기는 절반(28 → 14)** 이 됩니다. 4장에서는 이런 블록 두 개 뒤에 평탄화와 완전연결층을 붙여 CNN을 완성합니다.

**📝 1분 퀴즈** ① `[64, 32, 28, 28]` 특징 맵에 `nn.MaxPool2d(2, 2)` 를 쓰면 모양은? ② 풀링 층의 학습 파라미터는 몇 개?
<details><summary>정답 보기</summary>

① **[64, 32, 14, 14]** — 샘플 수와 채널 수는 그대로, 높이 · 너비만 절반
② **0개** — 최댓값(또는 평균)을 고르기만 하므로 학습할 가중치가 없습니다.
</details>

---
## 4. CNN 모델 만들기 — 강의 실습의 설계도  🎬 EP4

강의 실습과 **같은 구조**입니다. 강의노트 그림처럼 앞쪽은 특징 추출(Feature Extraction), 뒤쪽은 분류(Classification)입니다.

```
입력 [1, 28, 28]
 → 블록 1: Conv2d(1 → 32, 3×3, padding 1) → ReLU → MaxPool2d(2×2)    → [32, 14, 14]
 → 블록 2: Conv2d(32 → 64, 3×3, padding 1) → ReLU → MaxPool2d(2×2)   → [64, 7, 7]
 → 평탄화 → [3136] → Linear(3136 → 128) → ReLU → Linear(128 → 10)    → 숫자 10개의 점수
```

> 💡 설계도(`class`) · 부품 준비(`__init__`) · 순전파(`forward`) 문법은 지난 실습과 같습니다. 부품 이름(`conv1`, `pool1`, `fc1` …)은 나중에 필터와 특징 맵을 꺼내 볼 때 씁니다.

In [ ]:
class CNN(nn.Module):                                             # 합성곱 신경망 설계도
    def __init__(self):                                           # 부품 준비
        super().__init__()                                        # 부모 준비 — 항상 맨 먼저
        self.conv1 = nn.Conv2d(1, ___, kernel_size=3, padding=1)   # ✏️ 블록 1: 흑백 1 → 필터 수만큼
        self.relu1 = nn.ReLU()                                    # max(0, x)
        self.pool1 = nn._________(kernel_size=2, stride=2)        # ✏️ 28 → 14 (힌트: 최댓값 풀링 층)
        self.conv2 = nn.Conv2d(___, 64, kernel_size=3, padding=1)  # ✏️ 블록 2: 32 → 64채널
        self.relu2 = nn.ReLU()                                    # max(0, x)
        self.pool2 = nn._________(kernel_size=2, stride=2)        # ✏️ 14 → 7 (힌트: 최댓값 풀링 층)
        self.fc1 = nn.Linear(__________, 128)                     # ✏️ 평탄화 3136개 → 128
        self.relu3 = nn.ReLU()                                    # max(0, x)
        self.fc2 = nn.Linear(128, 10)                             # 128 → 숫자 10개의 점수

    def forward(self, x):                                         # 순전파: x 가 지나가는 길
        x = self.pool1(self.relu1(self.conv1(x)))                 # 블록 1: 합성곱 → ReLU → 풀링
        x = self.pool2(self.relu2(self.conv2(x)))                 # 블록 2
        x = x.view(x.size(0), -1)                                 # 평탄화 → [N, 3136]
        x = self.relu3(self.fc1(x))                               # 완전연결층 + ReLU
        return self.fc2(x)                                        # 점수 10개 (소프트맥스 전)

In [ ]:
torch.manual_seed(0)                       # 랜덤 초기 가중치를 '고정' (재현용)
model = CNN().to(device)                   # 설계도로 모델 생성 → 계산 장치(CPU/GPU)로
print(model)                               # 부품 목록 확인

### 4-1. 층마다 모양 따라가기 📐
1장에서 꺼낸 미니배치 `X` (64장)를 층 하나씩 통과시키며 모양을 출력합니다. `model.named_children()` 은 부품을 (이름, 층) 으로 차례로 꺼내 줍니다.

In [ ]:
h = X.to(device)                                  # 1장에서 꺼낸 미니배치  📐 [64, 1, 28, 28]
print(f"{'input':8s} {tuple(h.shape)}")           # 입력 모양
for name, layer in model.named_children():        # 부품을 순서대로 하나씩 꺼내
    if name == "fc1":                             # 완전연결층 바로 앞에서
        h = h.view(h.size(0), -1)                 # 평탄화 (forward 와 같은 줄)
        print(f"{'flatten':8s} {tuple(h.shape)}")  # 📐 [64, 3136]
    h = layer(h)                                  # 이 층 하나 통과
    print(f"{name:8s} {tuple(h.shape)}")          # 층 이름 · 출력 모양
check("최종 출력 모양", h.shape, (64, 10), tol=0)   # ✅ 64장 × 점수 10개

🔍 **결과 읽는 법** — ReLU 는 모양을 바꾸지 않고, 풀링만 크기를 절반으로 줄입니다. 합성곱(padding 1)은 크기를 지키고 채널만 바꿉니다. 그럼 이 크기들을 2장의 **공식**으로 미리 계산할 수 있을까요?

> 💡 **새 파이썬 문법 — 묶음 풀기 `for f, p, s in [...]`**
>
> 목록의 한 칸 `(3, 1, 1)` 을 꺼내 `f=3, p=1, s=1` 로 한 번에 나눠 담습니다.

In [ ]:
n = 28                                                    # MNIST 입력 크기
for f, p, s in [(3, 1, 1), (2, 0, 2), (3, 1, 1), (2, 0, 2)]:  # conv1 · pool1 · conv2 · pool2
    n = out_size(n, f, p, s)                              # 💡 (N − F + 2P) / S + 1
    print(n, end="  ")                                    # 한 줄에 이어서 출력
print("\n평탄화 크기 = 64 ×", n, "×", n, "=", 64 * n * n)  # fc1 의 입력 개수
check("공식으로 예측한 최종 크기", n, 7, tol=0)               # ✅ 28 → 28 → 14 → 14 → 7
check("평탄화 크기 64 × 7 × 7", 64 * n * n, 3136, tol=0)      # ✅ = nn.Linear(3136, 128)

### 4-2. 파라미터 개수 — 공식으로 세고, 코드로 확인
- **합성곱 층**: 필터 하나 = F × F × (입력 채널) 개의 가중치 + 편향 1개 → $(F \times F \times C_{in} + 1) \times C_{out}$
- **완전연결층**: (입력 수 + 1) × 출력 수 — 지난 실습과 같습니다.
- ReLU · 풀링 · 평탄화는 **0개** (학습할 가중치 없음)

> 💡 **새 파이썬 문법 — `sum(… for … in …)`**
>
> `for` 문으로 하나씩 더하는 일을 한 줄로 씁니다. `sum(p.numel() for p in layer.parameters())` = 층의 가중치 · 편향 개수의 합.

In [ ]:
formula = {"conv1": (3 * 3 * 1 + 1) * 32,        # 💡 (F×F×C_in + 1) × C_out
           "conv2": (3 * 3 * 32 + 1) * 64,       # 💡 같은 공식 (입력 채널 32)
           "fc1": (64 * 7 * 7 + 1) * 128,        # 💡 (입력 + 1) × 출력
           "fc2": (128 + 1) * 10}                # 💡 같은 공식
for name, layer in model.named_children():       # 부품마다
    n = sum(p.numel() for p in layer.parameters())  # 가중치 + 편향 개수 (ReLU·풀링은 0)
    print(f"{name:6s} {n:>9,}")                   # 이름과 개수 (천 단위 쉼표)
    if name in formula:                           # 파라미터가 있는 층이면
        check(f"{name} 공식 = 실제", n, formula[name], tol=0)  # ✅ 공식과 비교
total = sum(p.numel() for p in model.parameters())  # 모델 전체 파라미터
print(f"합계   {total:>9,}")                       # 전체 개수

### 4-3. 지난 실습의 다층 신경망(MLP)과 비교
지난 실습(FashionMNIST)의 MLP 는 784 → 512 → 512 → 10 이었습니다.
🔎 강의노트(실습)의 비교표에는 CNN 이 '약 100,000개'로 적혀 있습니다 — 코드가 센 값과 견주어 보세요.

In [ ]:
mlp = (784 + 1) * 512 + (512 + 1) * 512 + (512 + 1) * 10   # 지난 실습 MLP: 784→512→512→10
print(f"MLP {mlp:,}개 / CNN {total:,}개 → CNN 은 MLP 의 {total / mlp:.0%}")  # 개수 비교
print(f"CNN 중 fc1 한 층의 몫: {formula['fc1'] / total:.0%}")            # 대부분 fc1
check("지난 실습 MLP 파라미터", mlp, 669706, tol=0)    # ✅
check("CNN 전체 파라미터", total, 421642, tol=0)        # ✅

🔍 **결과 읽는 법** — 합성곱 층 두 개의 파라미터는 다 합쳐도 18,816 개(전체의 약 4 %)뿐입니다. 3×3 필터를 **모든 위치에서 같이 쓰기** 때문이죠. 대부분(95 %)은 평탄화 뒤의 `fc1` (3136 → 128) 한 층에 있습니다.
코드로 세어 보면 CNN 은 **421,642 개** — 강의노트 표의 어림값(약 100,000 개)보다는 많지만, MLP(669,706 개)의 63 % 로 더 적습니다. 그런데도 정확도는 더 높습니다(5장에서 확인).

### 4-4. 학습 전의 CNN은 숫자를 알아볼까?
모델의 출력은 숫자 10개 각각의 **점수**(logit)입니다. 가장 큰 점수의 번호가 예측이고(`argmax`), 점수를 확률로 바꾸는 것이 강의노트 그림 끝의 **소프트맥스(SoftMax)** 입니다.
학습에 쓸 `nn.CrossEntropyLoss` 가 안에서 소프트맥스를 해 주므로, 모델의 마지막에는 소프트맥스를 넣지 않습니다.

In [ ]:
with torch.no_grad():                                  # 평가만 할 때는 미분 추적 끄기
    scores = model(X.to(device)).cpu()                 # 📐 [64, 10] 샘플마다 점수 10개
probs = F.softmax(scores, dim=1)                       # 점수 → 확률 (샘플마다 합 = 1)
print("첫 샘플의 확률:", probs[0])                       # 모두 0.1 근처 → 아직 모름
pred = scores.argmax(1)                                # 가장 큰 점수의 번호 = 예측 숫자
print("예측:", pred[:10].tolist())                      # 앞 10개 예측
print("정답:", y[:10].tolist())                         # 앞 10개 정답
acc0 = (pred == y).float().mean().item()               # 맞힌 비율
check_range("학습 전 정확도 (찍기 ≈ 0.1)", acc0, 0.0, 0.3)          # ✅ 10개 중 하나 찍는 수준
check("확률의 합 = 1", probs.sum(dim=1), torch.ones(64), tol=1e-5)  # ✅ 소프트맥스

🔍 학습 전에는 열 개 숫자의 확률이 모두 0.1 근처라서, 사실상 **찍는 수준**입니다. 이제 학습으로 필터와 가중치 42만 개를 맞춰 나갑니다.

**📝 1분 퀴즈** ① `nn.Conv2d(32, 64, kernel_size=3)` 의 파라미터는 몇 개? ② `fc1` 의 입력이 3136 인 이유는?
<details><summary>정답 보기</summary>

① (3 × 3 × 32 + 1) × 64 = **18,496 개**
② 두 번째 풀링 뒤의 특징 맵이 **64 채널 × 7 × 7** 이고, 이것을 평탄화하면 3136 개이기 때문입니다.
</details>

---
## 5. 학습 · 평가 · 들여다보기  🎬 EP5

이 셀에서 할 일: 비용함수와 최적화기를 정하고, 4장과 **같은 초기 가중치**의 '새' CNN 을 만듭니다.
- **비용함수** `nn.CrossEntropyLoss()` : 숫자 10개 중 하나를 고르는 다중 분류용 교차엔트로피. 점수 → 확률(소프트맥스)까지 안에서 해 줍니다.
- **최적화기** `torch.optim.Adam` : 경사하강법을 개선한 방법 · 학습률 0.001 (강의 실습과 같음)

In [ ]:
loss_fn = nn.________________()                            # ✏️ 다중 분류 비용함수 (힌트: Cross…Loss)
torch.manual_seed(0)                                       # 4장과 같은 초기 가중치 (재현용)
model = CNN().to(device)                                   # 새 CNN — 처음부터 학습
optimizer = torch.optim.____(model.parameters(), lr=1e-3)  # ✏️ 최적화기, 학습률 0.001 (힌트: 4글자)

### 5-1. 학습 루프 4단계 — 지난 실습과 같은 뼈대

| 단계 | 코드 | 이론 |
|:-:|---|---|
| 1️⃣ 순전파 | `pred = model(X)` | 합성곱 → … → 점수 10개 |
| 2️⃣ 오차 계산 | `loss = loss_fn(pred, y)` | 교차엔트로피 |
| 3️⃣ 역전파 | `optimizer.zero_grad()` → `loss.backward()` | ∂E/∂w — 필터 가중치까지 전부 |
| 4️⃣ 업데이트 | `optimizer.step()` | w ← w − α · (Adam이 다듬은 기울기) |

`model.train()` / `model.eval()` 은 **학습 모드 / 평가 모드** 스위치입니다. 지금 모델은 어느 쪽이든 결과가 같지만, Dropout 같은 층을 넣으면 달라지므로 습관처럼 씁니다.

In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer):  # 학습 데이터 한 바퀴 = 1 에포크
    model.train()                                         # 학습 모드
    total, count = 0.0, 0                                 # 오차 합계, 샘플 수
    for X, y in loader:                                   # 미니배치(64장)마다
        X, y = X.to(device), y.to(device)                 # 계산 장치로 옮기기
        pred = model(X)                                   # 1️⃣ 순전파: 점수 10개
        loss = loss_fn(pred, y)                           # 2️⃣ 오차 (교차엔트로피)
        optimizer._________()                             # ✏️ 3️⃣ 이전 기울기 지우기 (힌트: 기울기 0)
        loss.________()                                   # ✏️ 3️⃣ 역전파: 42만 개의 기울기
        optimizer.____()                                  # ✏️ 4️⃣ 업데이트 (힌트: 한 걸음 이동)
        total += loss.item() * len(X)                     # 평균 오차 × 샘플 수 = 오차 합
        count += len(X)                                   # 샘플 수 세기
    return total / count                                  # 이번 에포크의 평균 오차

### 5-2. 평가 함수 — 시험 1만 장으로 채점
`pred.argmax(1)` 은 샘플마다(1번 축 방향으로) **점수가 가장 큰 번호** = 예측 숫자입니다. 정답 `y` 와 같은 개수를 셉니다.

In [ ]:
def evaluate(model, loader, loss_fn):                     # 정확도 · 평균 오차 계산
    model.eval()                                          # 평가 모드
    total, correct, count = 0.0, 0, 0                     # 오차 합계, 맞힌 수, 샘플 수
    with torch._______():                                 # ✏️ 평가만 할 때는 미분 추적 끄기
        for X, y in loader:                               # 미니배치마다
            X, y = X.to(device), y.to(device)             # 계산 장치로 옮기기
            pred = model(X)                               # 순전파만
            total += loss_fn(pred, y).item() * len(X)     # 오차 합계
            correct += (pred._________ == y).sum().item()  # ✏️ 예측 = 정답인 개수 (힌트: max의 번호)
            count += len(X)                               # 샘플 수
    return correct / count, total / count                 # 정확도, 평균 오차

### 5-3. 3 에포크 학습 ⏸
⏸ Colab CPU 에서는 에포크당 약 1분 30초, 3 에포크면 4~5분 걸립니다. GPU 런타임이면 훨씬 빠릅니다. ⏱ 걸린 시간도 출력합니다 (완성본에 찍힌 ⏱ 는 제작에 쓴 컴퓨터 기준).

> 💡 **새 파이썬 문법 — `time.time()`** : 지금 시각(초). 끝 시각 − 시작 시각 = 걸린 시간.

> ⚠️ 이 셀을 다시 실행하면 학습이 **이어서** 됩니다. 처음부터 다시 하려면 5장 첫 셀부터 다시 실행하세요.

In [ ]:
EPOCHS = 3                                                 # 학습 데이터를 3바퀴
history = {"train_loss": [], "test_loss": [], "test_acc": []}  # 에포크별 기록 상자
start = time.time()                                        # ⏱ 시작 시각
for epoch in range(1, EPOCHS + 1):                         # 에포크 1, 2, 3
    t0 = time.time()                                       # 이번 에포크 시작 시각
    tr_loss = train_one_epoch(model, train_loader, loss_fn, optimizer)  # 학습 한 바퀴
    te_acc, te_loss = evaluate(model, test_loader, loss_fn)            # 시험 채점
    history["train_loss"].append(tr_loss)                  # 기록: 학습 오차
    history["test_loss"].append(te_loss)                   # 기록: 시험 오차
    history["test_acc"].append(te_acc)                     # 기록: 시험 정확도
    print(f"에포크 {epoch}: 학습 오차 {tr_loss:.4f} | 시험 오차 {te_loss:.4f} | "  # 진행 상황
          f"정확도 {te_acc:.2%} | ⏱ {time.time() - t0:.0f}초")                     # (이어서)
print(f"⏱ 학습 전체 {time.time() - start:.0f}초")            # 총 학습 시간
check_range("시험 정확도 (목표 98.5 % 이상)", te_acc, 0.985, 1.0)  # ✅

🔍 **결과 읽는 법** — 시험 정확도는 1 에포크 98.1 % → 2 에포크 98.8 % → 3 에포크 **98.9 %** 입니다(이 노트북을 만든 컴퓨터의 CPU 기준). 시험 1만 장 중 약 110 장만 틀린 셈이죠.
이 노트북을 만들 때 씨앗(seed)을 바꿔 네 번 학습해 보니, 1 에포크는 네 번 모두 98.5 % 에 조금 못 미쳤고 2 에포크는 네 번 중 세 번, **3 에포크는 네 번 모두 98.5 % 를 넘었습니다**. 그래서 3 에포크로 정했습니다.
(GPU · Colab 에서는 소수점 아래 숫자가 조금 다를 수 있습니다)

In [ ]:
ep = list(range(1, EPOCHS + 1))                            # x축: 에포크 1, 2, 3
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.6))    # 그래프 2개를 가로로
ax1.plot(ep, history["train_loss"], "o-", label="train")   # 학습 오차
ax1.plot(ep, history["test_loss"], "s-", label="test")     # 시험 오차
ax2.plot(ep, [v * 100 for v in history["test_acc"]], "o-")  # 시험 정확도 (%)
ax1.set_title("Loss"); ax1.set_xlabel("epoch"); ax1.legend()            # 꾸미기 (영문)
ax2.set_title("Test accuracy (%)"); ax2.set_xlabel("epoch")             # 꾸미기
ax1.set_xticks(ep); ax2.set_xticks(ep)                     # 눈금: 1, 2, 3
plt.tight_layout()                                         # 간격 정리
plt.show()                                                 # 그리기

### 5-4. 들여다보기 ① — 학습된 필터 32개
`model.conv1.weight` 의 모양은 `[32, 1, 3, 3]` — 3×3 필터 32개입니다(밝을수록 +, 어두울수록 −).
3장에서 손으로 만든 **세로 경계 필터**(왼쪽 어둡고 오른쪽 밝음)와 비슷한 모양이 있나요? 가로 경계 · 대각선 모양도 찾아보세요.

In [ ]:
w = model.conv1.weight.detach().cpu()                 # 📐 [32, 1, 3, 3] 학습된 필터 32개
fig, axes = plt.subplots(4, 8, figsize=(10, 5.4))     # 4줄 × 8칸
for i, ax in enumerate(axes.flat):                    # 필터 하나씩
    ax.imshow(w[i, 0], cmap="gray")                   # 3×3 필터 그리기
    ax.set_title(f"filter {i}", fontsize=8)           # 필터 번호
    ax.axis("off")                                    # 눈금 감추기
plt.suptitle("conv1 filters (3x3) after training")    # 전체 제목 (영문)
plt.tight_layout()                                    # 간격 정리
plt.show()                                            # 그리기
check("conv1 필터 모양 (32, 1, 3, 3)", w.shape, (32, 1, 3, 3), tol=0)  # ✅

### 5-5. 들여다보기 ② — 숫자 7 의 특징 맵
3장에서 쓴 시험 데이터 첫 장(숫자 7)을 학습된 CNN 에 넣고, 블록 1 · 블록 2 의 특징 맵을 꺼내 봅니다. (그림을 그리는 코드는 결과만 보셔도 됩니다)

In [ ]:
model.eval()                                          # 평가 모드
with torch.no_grad():                                 # 계산기 모드
    d = digit.view(1, 1, 28, 28).to(device)           # 숫자 7 (1장짜리 미니배치)
    f1 = model.relu1(model.conv1(d))                  # 블록 1 의 특징 맵  📐 [1, 32, 28, 28]
    f2 = model.relu2(model.conv2(model.pool1(f1)))    # 블록 2 의 특징 맵  📐 [1, 64, 14, 14]
f1, f2 = f1.cpu(), f2.cpu()                           # 그리기 위해 CPU 로
check("블록 1 특징 맵 모양 (필터 32개 → 32장)", f1.shape, (1, 32, 28, 28), tol=0)  # ✅
check("블록 2 특징 맵 모양 (필터 64개 → 64장)", f2.shape, (1, 64, 14, 14), tol=0)  # ✅

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(12, 3.4))     # 2줄 × 8칸
for i in range(8):                                    # 앞에서 8장씩
    axes[0, i].imshow(f1[0, i], cmap="viridis")       # 윗줄: 블록 1 특징 맵 (28×28)
    axes[1, i].imshow(f2[0, i], cmap="viridis")       # 아랫줄: 블록 2 특징 맵 (14×14)
    axes[0, i].set_title(f"conv1 map {i}", fontsize=8)  # 제목 (영문)
    axes[1, i].set_title(f"conv2 map {i}", fontsize=8)  # 제목
for ax in axes.flat:                                  # 모든 칸
    ax.axis("off")                                    # 눈금 감추기
plt.tight_layout()                                    # 간격 정리
plt.show()                                            # 그리기

🔍 **결과 읽는 법** — 블록 1 의 특징 맵은 아직 숫자 7 의 모양이 그대로 보이고, 필터마다 **가로 획 · 대각선 획 · 경계 · 배경**처럼 서로 다른 부분이 밝게 나타납니다(3장의 손 필터가 한 일과 같음). 블록 2 의 특징 맵은 14×14 로 작아지고, 7 의 꺾이는 모서리나 획의 한 부분처럼 **특정한 모양이 있는 곳에만** 반응합니다. 층이 깊어질수록 더 추상적인 특징을 본다는 것이 강의노트의 'Feature Learning' 입니다.

### 5-6. 틀린 숫자 모아 보기
시험 1만 장 중 CNN 이 틀린 그림만 모아 봅니다. 어떤 숫자끼리 헷갈렸을까요?

In [ ]:
wrong_img, wrong_true, wrong_pred = [], [], []          # 틀린 그림 · 정답 · 예측을 모을 목록
model.eval()                                            # 평가 모드
with torch.no_grad():                                   # 계산기 모드
    for X_b, y_b in test_loader:                        # 시험 미니배치마다
        p_b = model(X_b.to(device)).argmax(1).cpu()     # 예측 숫자
        bad = p_b != y_b                                # 틀린 자리만 True
        wrong_img.append(X_b[bad])                      # 틀린 그림 모으기
        wrong_true.append(y_b[bad])                     # 그 정답
        wrong_pred.append(p_b[bad])                     # 그 예측
wrong_img = torch.cat(wrong_img)                        # 목록 → 텐서 하나로 이어 붙이기
wrong_true, wrong_pred = torch.cat(wrong_true), torch.cat(wrong_pred)  # 정답 · 예측도
print("틀린 개수:", len(wrong_img), "/ 10000")           # 1만 장 중 몇 장?
check("틀린 개수 = (1 − 정확도) × 10000", len(wrong_img), round((1 - te_acc) * 10000), tol=2)  # ✅

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(10, 3.9))       # 2줄 × 6칸 = 12장
for i, ax in enumerate(axes.flat):                      # 틀린 그림 앞 12장
    ax.imshow(wrong_img[i, 0], cmap="gray")             # 그림
    ax.set_title(f"true {wrong_true[i]} / pred {wrong_pred[i]}", fontsize=9)  # 정답 / 예측
    ax.axis("off")                                      # 눈금 감추기
plt.tight_layout()                                      # 간격 정리
plt.show()                                              # 그리기

🔍 틀린 그림 중에는 사람이 봐도 헷갈리는 글씨(획이 끊긴 숫자, 4 와 9, 3 과 5 처럼 닮은 숫자)가 많습니다. CNN 의 실수가 '이해할 만한' 실수라는 것도 중요한 확인입니다.

**📝 1분 퀴즈** ① 평가할 때 `with torch.no_grad():` 를 쓰는 이유는? ② 정확도가 98.9 % 이면 시험 1만 장 중 대략 몇 장을 틀린 걸까요?
<details><summary>정답 보기</summary>

① 평가에는 기울기가 필요 없으므로 **미분 추적을 꺼서** 계산을 빠르게 하고 메모리를 아낍니다.
② 1만 × 1.1 % ≈ **110 장**
</details>

### 5-7. 모델 저장 · 불러오기 · 예측 ✅
학습된 가중치(`state_dict`)를 파일로 저장해 두면, 다음에 다시 학습하지 않고 바로 쓸 수 있습니다. 새 모델에 불러와서 정확도가 **같은지** 확인합니다.

In [ ]:
torch.save(model.state_dict(), "mnist_cnn.pth")            # 학습된 가중치를 파일로 저장
model2 = CNN().to(device)                                  # 새 모델 (아직 랜덤 가중치)
model2.load_state_dict(torch.load("mnist_cnn.pth", map_location=device))  # 저장한 가중치 넣기
acc2, _ = evaluate(model2, test_loader, loss_fn)           # 불러온 모델로 시험 (_ = 안 쓰는 값)
print(f"원래 모델 {te_acc:.2%} / 불러온 모델 {acc2:.2%}")    # 같아야 정상
check("불러온 모델의 정확도 = 원래 정확도", acc2, te_acc, tol=0.001)  # ✅

In [ ]:
imgs = torch.stack([test_data[i][0] for i in range(10)])   # 시험 그림 앞 10장  📐 [10, 1, 28, 28]
labels = [test_data[i][1] for i in range(10)]              # 그 정답 10개
with torch.no_grad():                                      # 계산기 모드
    preds = model2(imgs.to(device)).argmax(1).tolist()     # 불러온 모델로 예측
print("예측:", preds)                                       # 모델이 고른 숫자
print("정답:", labels)                                      # 실제 숫자

### 5-8. (선택) 한 줄만 바꿔 FashionMNIST 로 — 지난 실습 MLP 와 비교 ⏸
**선택 실습**입니다. 데이터 이름 **한 줄**만 `datasets.FashionMNIST` 로 바꾸면, 같은 CNN 설계도 · 같은 학습 함수 · 같은 3 에포크로 옷 사진 10종을 분류합니다.
지난 실습(FashionMNIST)의 다층 신경망(MLP)은 설정을 잘 맞추면 **약 85~88 %** 였습니다. ⏸ Colab CPU 에서 5분쯤 걸립니다.

In [ ]:
DATASET = datasets.FashionMNIST            # 🔁 바꾼 곳은 이 한 줄! (datasets.MNIST → FashionMNIST)
f_train = DATASET(root="data", train=True, download=True, transform=ToTensor())   # 6만 장
f_test = DATASET(root="data", train=False, download=True, transform=ToTensor())   # 1만 장
f_train_loader = DataLoader(f_train, batch_size=BATCH, shuffle=True,              # 1장과 같은 설정
                            generator=torch.Generator().manual_seed(0))           # 섞는 순서 고정
f_test_loader = DataLoader(f_test, batch_size=BATCH, shuffle=False)               # 시험용
print("FashionMNIST:", len(f_train), "/", len(f_test), "장")  # 6만 / 1만 (모양도 MNIST 와 같음)

In [ ]:
torch.manual_seed(0)                                       # 같은 초기 가중치
f_model = CNN().to(device)                                 # 같은 설계도 (4장)
f_opt = torch.optim.Adam(f_model.parameters(), lr=1e-3)    # 같은 최적화기 · 학습률
t0 = time.time()                                           # ⏱ 시작 시각
for epoch in range(1, EPOCHS + 1):                         # 같은 3 에포크
    train_one_epoch(f_model, f_train_loader, loss_fn, f_opt)  # 같은 학습 함수
    f_acc, _ = evaluate(f_model, f_test_loader, loss_fn)   # 같은 평가 함수
    print(f"에포크 {epoch}: FashionMNIST 시험 정확도 {f_acc:.2%}")  # 진행 상황
print(f"⏱ {time.time() - t0:.0f}초 · MNIST {te_acc:.2%} / FashionMNIST {f_acc:.2%}")  # 비교
check_range("FashionMNIST CNN 정확도 (MLP 85~88 % 보다 높게)", f_acc, 0.88, 0.95)  # ✅

🔍 **결과 읽는 법** — 같은 CNN · 같은 코드인데 MNIST 는 98.9 %, FashionMNIST 는 90.9 % 입니다. 셔츠 · 티셔츠 · 풀오버처럼 **서로 닮은 옷**이 손글씨 숫자보다 어렵기 때문이죠. 그래도 CNN 은 1 에포크(88.3 %)만에 지난 실습 MLP 의 최고 수준(약 85~88 %)에 닿고, 3 에포크에 90.9 % 로 앞섭니다 — 파라미터는 MLP 의 63 % 인데도요. 강의노트 실습의 기대 결과(10 에포크에 91~92 %)와도 가깝습니다.

### 5-9. 🏆 도전 과제
1. **필터 수 줄이기** — `conv1`, `conv2` 의 필터 수를 (16, 32) 로 줄이면 파라미터 수와 정확도는? (힌트: `conv2` 의 입력 채널과 `fc1` 의 입력 `32 * 7 * 7` 도 함께 고치기 · 개수는 4-2 의 공식으로 먼저 계산)
2. **패딩 없이** — 두 합성곱 층을 `padding=0` 으로 바꾸면 크기가 28 → 26 → 13 → 11 → 5 로 줄어듭니다. `fc1` 의 입력 크기는 얼마여야 할까요? `out_size()` 로 계산한 뒤 `nn.Linear` 를 고쳐 학습해 보세요.
3. **풀링 대신 stride 2** — `MaxPool2d` 를 빼고 합성곱 층을 `stride=2` 로 바꾸면(강의노트 과제) 크기 · 파라미터 수 · 정확도가 어떻게 달라질까요?

---
## 6. 핵심 요약

### 이론 ↔ 코드 대응표 (CNN 판)
| 이론 (강의노트) | PyTorch 코드 |
|---|---|
| 흑백 그림 · [N, C, H, W] | `X.shape` → `[64, 1, 28, 28]` |
| 합성곱 (필터를 밀며 곱해서 더하기) | `F.conv2d(x, k)` · `nn.Conv2d(1, 32, kernel_size=3)` |
| 필터 크기 F · 필터 수 | `kernel_size=3` · 두 번째 인자 `32` (= 특징 맵 수) |
| 스트라이드 S | `stride=2` |
| 제로 패딩 P | `padding=1` · `F.pad(x, (1, 1, 1, 1))` |
| 출력 크기 (N − F + 2P) / S + 1 | `out_size(n, f, p, s)` |
| 활성화 ReLU = max(0, x) | `nn.ReLU()` · `F.relu(x)` |
| 최댓값 · 평균값 풀링 | `nn.MaxPool2d(kernel_size=2, stride=2)` · `F.max_pool2d` · `F.avg_pool2d` |
| 평탄화 (Flatten) | `x.view(x.size(0), -1)` |
| 완전연결층 | `nn.Linear(64 * 7 * 7, 128)` |
| 소프트맥스 + 교차엔트로피 | `nn.CrossEntropyLoss()` |
| 경사하강법 (Adam) | `torch.optim.Adam(model.parameters(), lr=1e-3)` |
| 파라미터 수 (F·F·C_in + 1) · C_out | `sum(p.numel() for p in model.parameters())` |

### 학습 루프 4단계 (지난 실습과 같은 뼈대!)
```python
pred = model(X)            # 1️⃣ 순전파
loss = loss_fn(pred, y)    # 2️⃣ 오차 계산
optimizer.zero_grad()      # 3️⃣ 역전파: 기울기 지우고
loss.backward()            #           기울기 계산
optimizer.step()           # 4️⃣ 가중치 업데이트
```

### ⚠️ 자주 하는 실수 5가지
1. 그림 1장을 `[1, 28, 28]` 그대로 모델에 넣음 → 합성곱까지는 지나가지만 평탄화에서 모양이 꼬여 `mat1 and mat2 shapes cannot be multiplied (64x49 …)` 오류. 1장도 `.view(1, 1, 28, 28)` 처럼 **[N, C, H, W]** '1장짜리 미니배치'로!
2. 평탄화를 `x.view(-1)` 로 함 → 64장이 한 줄로 섞임. `x.view(x.size(0), -1)` 로 샘플 수를 지키기
3. 층 구조(패딩 · 풀링 · 필터 수)를 바꾸고 `fc1` 의 입력 크기를 그대로 둠 → `mat1 and mat2 shapes cannot be multiplied` 오류. 출력 크기 공식으로 다시 계산!
4. 모델은 GPU, 데이터는 CPU → `Expected all tensors to be on the same device` 오류. 데이터도 `.to(device)`
5. `optimizer.zero_grad()` 를 빼먹음 → 기울기가 누적되어 엉뚱하게 학습

### 강의노트의 CNN 실습 코드와 무엇이 다른가요?
- 데이터: FashionMNIST → **MNIST** (마지막 선택 실습에서 한 줄만 바꿔 FashionMNIST 와 비교)
- 모델 구조 · 비용함수 · 최적화기(Adam, lr = 0.001) · 미니배치 64 는 **그대로**
- 10 에포크 → **3 에포크** (MNIST 는 3 에포크면 98.5 % 이상 · Colab CPU 에서도 4~5분)
- 씨앗(seed) 고정 · ✅ 자가검증 · 강의노트 그림의 숫자 재현을 더했습니다.

### 다음 실습 예고
이미지 다음은 **순서가 있는 데이터(시계열)** 입니다. 다음 실습에서는 시계열을 다루는 순환 신경망(RNN)을 PyTorch로 만듭니다.